In [1]:
!mkdir -p /content/imsitu_dataset
!wget -nc https://raw.githubusercontent.com/my89/imSitu/refs/heads/master/train.json -P /content/imsitu_dataset
!wget -nc https://raw.githubusercontent.com/my89/imSitu/refs/heads/master/dev.json -P /content/imsitu_dataset
!wget -nc https://raw.githubusercontent.com/my89/imSitu/refs/heads/master/test.json -P /content/imsitu_dataset


--2025-12-04 03:06:08--  https://raw.githubusercontent.com/my89/imSitu/refs/heads/master/train.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22220963 (21M) [text/plain]
Saving to: ‘/content/imsitu_dataset/train.json’

train.json          100%[===================>]  21.19M  --.-KB/s    in 0.08s   

2025-12-04 03:06:09 (259 MB/s) - ‘/content/imsitu_dataset/train.json’ saved [22220963/22220963]

--2025-12-04 03:06:09--  https://raw.githubusercontent.com/my89/imSitu/refs/heads/master/dev.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 20

In [2]:
import nltk

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
import json
from pathlib import Path

IMSITU_PATH = Path("/content/imsitu_dataset")

with open(IMSITU_PATH / "train.json", "r") as f:
    imsitu_train = json.load(f)

In [ ]:
from nltk.corpus import wordnet as wn

def get_primary_word(offset):
    synset = wn.synset_from_pos_and_offset('n', int(offset))
    lemmas = synset.lemma_names()
    primary_word = lemmas[0]  # Often the most common word

    return primary_word

male_agents = {
        'man', 'men', 'male', 'boy', 'boys', 'guy', 'guys',
        'father', 'dad', 'daddy', 'husband', 'son', 'brother', 
        'grandfather', 'grandpa', 'uncle', 'nephew',
        'groom', 'prince', 'king', 'monk', 'priest', 'pope',
        'waiter', 'actor', 'policeman', 'fireman', 'businessman', 'businessmen',
        'cowboy', 'chairman', 'postman', 'mailman', 'gentleman', 'sir',
        'lad', 'patriarch', 'masseur', 'host', 'hero',
        'black_man', 'white_man', 'middle-aged_man', 'old_man', 
        'young_buck', 'male_child', 'gay_man', 'man_of_means', 'man_of_action'
}
female_agents = {
        'woman', 'women', 'female', 'girl', 'girls', 'lady', 'ladies',
        'mother', 'mom', 'mommy', 'wife', 'daughter', 'sister', 
        'grandmother', 'grandma', 'aunt', 'niece',
        'bride', 'princess', 'queen', 'nun', 'maid', 
        'waitress', 'actress', 'businesswoman', 'cowgirl', 'chairwoman',
        'hostess', 'masseuse', 'heroine', 'housewife', 
        'black_woman', 'white_woman', 'old_woman', 'old_lady', 
        'female_child', 'lass', 'dame', 'flower_girl', 'womanhood', 
        'oarswoman', 'dairymaid', 'bridesmaid', 'lioness', 'leopardess'
}

def get_agent_gender(agent):
    if agent in male_agents:
        return "M"
    if agent in female_agents:
        return "F"
    return None

In [5]:
def save_annotated_json(split="train"):
    with open(IMSITU_PATH / f"{split}.json", "r") as f:
        dataset = json.load(f)
        
    imsitu_annotated = list()

    for image_name, details in dataset.items():
        frames = details.get('frames', [])
        if not frames:
            continue
        agent_wordnet = frames[0].get('agent', '')
        if not agent_wordnet:
            continue
        agent = get_primary_word(agent_wordnet.split('n')[1])
        gender = get_agent_gender(agent)
        if not gender:
            continue
        
        entry = {
            'verb': details['verb'],
            'gender': gender, # 0 or 1
            'agent_word': agent,
            "image_path": image_name
        }

        imsitu_annotated.append(entry)

    with open(IMSITU_PATH / f"imsitu_annotated_{split}.json", "w") as f:
        json.dump(imsitu_annotated, f)

In [6]:
save_annotated_json("train")
save_annotated_json("dev")
save_annotated_json("test")

In [7]:
!cat "$IMSITU_PATH/imsitu_annotated_dev.json" | jq -M

[
  {
    "verb": "hitchhiking",
    "gender": "M",
    "agent_word": "man",
    "image_path": "hitchhiking_238.jpg"
  },
  {
    "verb": "puckering",
    "gender": "M",
    "agent_word": "man",
    "image_path": "puckering_158.jpg"
  },
  {
    "verb": "standing",
    "gender": "M",
    "agent_word": "man",
    "image_path": "standing_66.jpg"
  },
  {
    "verb": "boarding",
    "gender": "M",
    "agent_word": "man",
    "image_path": "boarding_168.jpg"
  },
  {
    "verb": "putting",
    "gender": "M",
    "agent_word": "man",
    "image_path": "putting_27.jpg"
  },
  {
    "verb": "twisting",
    "gender": "F",
    "agent_word": "woman",
    "image_path": "twisting_117.jpg"
  },
  {
    "verb": "pooing",
    "gender": "M",
    "agent_word": "man",
    "image_path": "pooing_33.jpg"
  },
  {
    "verb": "photographing",
    "gender": "F",
    "agent_word": "woman",
    "image_path": "photographing_16.jpg"
  },
  {
    "verb": "lighting",
    "gender": "F",
    "agent_word": "woman",


In [8]:
!wget https://s3.amazonaws.com/my89-frame-annotation/public/of500_images_resized.tar

--2025-12-04 03:06:22--  https://s3.amazonaws.com/my89-frame-annotation/public/of500_images_resized.tar
Resolving s3.amazonaws.com (s3.amazonaws.com)... 3.5.22.32, 16.15.203.205, 16.15.216.20, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|3.5.22.32|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4078049280 (3.8G) [application/x-tar]
Saving to: ‘of500_images_resized.tar’

of500_images_resize 100%[===================>]   3.80G  34.5MB/s    in 1m 59s  

2025-12-04 03:08:21 (32.6 MB/s) - ‘of500_images_resized.tar’ saved [4078049280/4078049280]



In [9]:
!tar -xf of500_images_resized.tar -C "$IMSITU_PATH"

In [10]:
!ls "$IMSITU_PATH"

dev.json		    imsitu_annotated_train.json  train.json
imsitu_annotated_dev.json   of500_images_resized
imsitu_annotated_test.json  test.json


In [11]:
import random
import matplotlib.pyplot as plt
from PIL import Image
import os

male_annots = [
    [k, *v.values()]
    for k, v in imsitu_annotated.items()
    if v['gender'] == "M"
]

female_annots = [
    [k, *v.values()]
    for k, v in imsitu_annotated.items()
    if v['gender'] == "F"
]

random_males = random.sample(male_annots, 10)
random_females = random.sample(female_annots, 10)

def visualize_samples(samples, title):
    plt.figure(figsize=(18, 8))
    plt.suptitle(title, fontsize=20)

    for idx, (fname, verb, gender, agent) in enumerate(samples):
        img_path = os.path.join("/content/imsitu_dataset/of500_images_resized", fname)
        img = Image.open(img_path)

        plt.subplot(2, 5, idx + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{verb} ({agent})", fontsize=10)

    plt.tight_layout()
    plt.show()

visualize_samples(random_males, "Random 10 Male Samples")
visualize_samples(random_females, "Random 10 Female Samples")

NameError: name 'imsitu_annotated' is not defined